In [ ]:
# ==============================================================================
# LAB 6: NLP FUNDAMENTALS — TOKENIZATION, EMBEDDINGS & TEXT CHUNKING
# Complete, runnable Google Colab implementation for Tasks 1, 2, and 3
# ==============================================================================

# Step 0: Fix Colab's pre-installed 'peft' version conflict and install packages
!pip uninstall -y -q peft
!pip install -q -U scikit-learn sentence-transformers transformers numpy

import sys
# Clear cached modules from any previous failed import so a runtime restart isn't needed
for mod in list(sys.modules.keys()):
    if mod.startswith(("peft", "transformers", "sentence_transformers")):
        del sys.modules[mod]

import re
import csv
import json
import time
import math
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# Load model onto CPU as required by lab specification
print("[Setup] Loading SentenceTransformer model...")
encoder_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
print("[Setup] Model loaded successfully.\n")

# ==============================================================================
# SECTION 1: TASK 1 — TOKENIZATION & FAQ CHATBOT
# ==============================================================================
print("=" * 75)
print("TASK 1: TOKENIZATION COMPARISON (WHITESPACE vs. REGEX vs. SUBWORD)")
print("=" * 75)

sample_inputs = [
    "AI, search!",
    "I can't log in.",
    "help@example.com",
    "CS4944 version 2.1",
    "",
    "یہ ایک مثال ہے"  # Urdu test text
]

for text in sample_inputs:
    ws_tokens = text.split()
    word_tokens = re.findall(r"\b\w+\b", text, flags=re.UNICODE)

    # Subword tokenizer via model tokenizer
    if text.strip():
        subwords = encoder_model.tokenizer.tokenize(text)
    else:
        subwords = []

    print(f"Input Text    : {repr(text)}")
    print(f"  Whitespace  ({len(ws_tokens):>2}): {ws_tokens}")
    print(f"  Word-like   ({len(word_tokens):>2}): {word_tokens}")
    print(f"  Subword     ({len(subwords):>2}): {subwords}")
    print("-" * 75)

# FAQ Bot Implementation (Expanded with +2 FAQs per task instruction)
FAQ_DATA = [
    {"id": "faq:password", "question": "How do I reset my password?", "answer": "Use the recovery link on the login page.", "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?", "answer": "Use the assignment upload page in the learning portal.", "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?", "answer": "Request an extension with medical evidence before the deadline.", "source": "handbook.txt"},
    # Added FAQs
    {"id": "faq:wifi", "question": "How do I connect to campus WiFi?", "answer": "Select Campus-Secure and authenticate with your student ID.", "source": "network.txt"},
    {"id": "faq:fee", "question": "What is the fee payment deadline?", "answer": "Tuition fees must be cleared within 10 days of semester start.", "source": "finance.txt"}
]

class FAQBot:
    def __init__(self, faq_list, backend="tfidf"):
        self.faq_list = faq_list
        self.backend = backend
        self.texts = [item["question"] + " " + item["answer"] for item in faq_list]
        self.vectorizer = TfidfVectorizer()
        self.matrix = self.vectorizer.fit_transform(self.texts)

    def search_candidates(self, query, k=3):
        q_vec = self.vectorizer.transform([query])
        if q_vec.nnz == 0:
            return []
        scores = (q_vec @ self.matrix.T).toarray()[0]
        ranked_idx = np.argsort(-scores)
        hits = [(self.faq_list[i], float(scores[i])) for i in ranked_idx[:k] if scores[i] > 0]
        return hits

    def reply(self, query):
        if not query.strip():
            return "Please enter a question."
        # Canonical match check
        for item in self.faq_list:
            if query.strip().casefold() == item["question"].casefold():
                return item["answer"] + " [" + "source: " + item["source"] + "]"
        # Fallback to ranked suggestions
        hits = self.search_candidates(query)
        if not hits:
            return "No matching FAQ. Please contact the help desk."
        output = ["Possible matches; choose or rephrase your question:"]
        for item, score in hits:
            output.append(f" - {item['question']} (similarity {score:.3f})")
        return "\n".join(output)

faq_engine = FAQBot(FAQ_DATA)
print("\n--- Task 1 FAQ Bot Test Runs ---")
print("1. Exact Match  :", faq_engine.reply("How do I reset my password?"))
print("2. Paraphrase   :\n" + faq_engine.reply("password recovery link"))
print("3. Unsupported  :", faq_engine.reply("volcano geology"))
print("4. Empty Query  :", faq_engine.reply(""))


# ==============================================================================
# SECTION 2: TASK 2 — CHUNKING MECHANISMS & BOUNDARY VALIDATION
# ==============================================================================
print("\n" + "=" * 75)
print("TASK 2: CHUNKING ALGORITHMS & BOUNDARY AUDITS")
print("=" * 75)

def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]

def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not (0 <= overlap < size):
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    if not spans:
        return []
    result = []
    step = size - overlap
    for i in range(0, len(spans), step):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j-1][1]))
        if j == len(spans):
            break
    return result

def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    cuts = [0] + [m.end() for m in re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    return out

def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        if strategy == "sentence":
            spans = sentence_chunks(text, size=40)
        else:
            spans = windows(text, size=40, overlap=int(strategy))
        for i, (a, b) in enumerate(spans):
            result.append({
                "id": f"{doc_id}:{i}",
                "doc": doc_id,
                "start": a,
                "end": b,
                "text": text[a:b]
            })
    return result

# Boundary test suite (Empty, Short, Invalid overlap, Oversized sentence)
print("Executing Task 2 boundary tests...")
assert windows("") == []
assert windows("short text", size=40, overlap=10) == [(0, 10)]

try:
    windows("invalid overlap text", size=10, overlap=10)
    raise AssertionError("Failed: invalid overlap was accepted!")
except ValueError:
    print(" - Assertion passed: Invalid overlap (overlap >= size) properly rejected.")

long_sentence = " ".join(["token"] * 95) + "."
long_spans = sentence_chunks(long_sentence, size=40)
assert all(len(long_sentence[a:b].split()) <= 40 for a, b in long_spans)
print(f" - Assertion passed: 95-word sentence properly partitioned into {len(long_spans)} fallback chunks under budget 40.")


# ==============================================================================
# SECTION 3: TASK 3 — RETRIEVAL BENCHMARK, AUDIT & LOGGING
# ==============================================================================
print("\n" + "=" * 75)
print("TASK 3: 5-DOC CORPUS, TF-IDF VS. EMBEDDINGS (HIT@3 & MRR@3 EVALUATION)")
print("=" * 75)

# 1. Build Synthetic Corpus (5 expanded documents with rule/exception pairs)
docs = {
    "late": (
        "Late submission is normally penalized by ten marks per day. "
        "Illness extensions require medical evidence before the deadline. "
        "Students must submit verified hospital forms to the department coordinator. "
        "Unapproved late assignments will receive an automatic zero mark."
    ),
    "password": (
        "Reset a forgotten password using the recovery link on the primary domain. "
        "Never share recovery codes with third parties or IT support personnel. "
        "If multi-factor authentication fails, visit room 302 with university photo ID. "
        "Accounts remain locked after five incorrect attempts."
    ),
    "assignment": (
        "Upload assignments through the learning portal before midnight. "
        "Submit the final file as a PDF without password protection. "
        "Ensure file size does not exceed twenty megabytes. "
        "Zip archives and external drive links are not accepted."
    ),
    "contractor": (
        "Contractors can work remotely only if the signed contract permits it. "
        "Employee rules do not apply to contractors under standard consultancy terms. "
        "All offsite connections must use the approved institutional VPN profile. "
        "Security breaches lead to immediate contract termination."
    ),
    "printer": (
        "For blank printer pages, check ink levels and run the cleaning utility. "
        "Print a test page afterwards to verify toner alignment. "
        "Do not pull jammed paper while the feeder rollers are engaged. "
        "Contact lab technicians if error code 50 persists."
    )
}

# 2. Queries: 8 Answerable (4 dev, 4 test) + 2 Unanswerable (test)
questions_raw = [
    # Dev Answerable
    ("What is required for illness extensions?", "late", "Illness extensions require medical evidence before the deadline.", "dev"),
    ("How do I reset a forgotten password?", "password", "Reset a forgotten password using the recovery link on the primary domain.", "dev"),
    ("Where do I upload assignments?", "assignment", "Upload assignments through the learning portal before midnight.", "dev"),
    ("Can contractors work remotely?", "contractor", "Contractors can work remotely only if the signed contract permits it.", "dev"),
    # Test Answerable
    ("What do I do for blank printer pages?", "printer", "For blank printer pages, check ink levels and run the cleaning utility.", "test"),
    ("How can I recover my login access?", "password", "Reset a forgotten password using the recovery link on the primary domain.", "test"),
    ("How should I hand in my coursework?", "assignment", "Upload assignments through the learning portal before midnight.", "test"),
    ("What proof is needed when I was ill?", "late", "Illness extensions require medical evidence before the deadline.", "test"),
    # Test Unanswerable
    ("What are tomorrow's flight prices?", None, None, "test"),
    ("Who won the football match?", None, None, "test")
]

queries = [
    {"query": q, "doc": d, "evidence": e, "split": s}
    for q, d, e, s in questions_raw
]

# Write out corpus and queries
Path("corpus.json").write_text(json.dumps(docs, indent=2), encoding="utf-8")
Path("queries.json").write_text(json.dumps(queries, indent=2), encoding="utf-8")

def evaluate_metrics(scores_matrix, chunks, eval_items):
    hits, reciprocal_ranks = [], []
    for row, q in zip(scores_matrix, eval_items):
        if q["doc"] is None:
            continue  # Exclude unanswerable queries from Hit/MRR
        top_indices = np.argsort(-row, kind="stable")[:3]

        # Ground truth check: correct doc AND exact evidence text contained in chunk
        correct_ranks = [
            rank for rank, idx in enumerate(top_indices, 1)
            if chunks[idx]["doc"] == q["doc"] and q["evidence"] in chunks[idx]["text"]
        ]
        hits.append(bool(correct_ranks))
        reciprocal_ranks.append(1.0 / correct_ranks[0] if correct_ranks else 0.0)

    return (float(np.mean(hits)), float(np.mean(reciprocal_ranks))) if hits else (0.0, 0.0)

query_texts = [q["query"] for q in queries]
strategies = ["0", "10", "sentence"]
methods = ["tfidf", "embedding"]
logs = []

print(f"{'Strategy':<10} | {'Method':<10} | {'Split':<6} | {'Hit@3':<7} | {'MRR@3':<7} | {'Chunks':<6} | {'Enc(s)':<8} | {'Srch(s)'}")
print("-" * 80)

for strat in strategies:
    chunks = make_chunks(docs, strat)
    chunk_texts = [c["text"] for c in chunks]

    # Audit model sequence token lengths
    lengths = [len(encoder_model.tokenizer(t, truncation=False)["input_ids"]) for t in chunk_texts + query_texts]
    if max(lengths) > encoder_model.max_seq_length:
        raise ValueError(f"Max token length {max(lengths)} exceeds model limit {encoder_model.max_seq_length}")

    for method in methods:
        t_enc_start = time.perf_counter()
        if method == "tfidf":
            vec = TfidfVectorizer()
            C = vec.fit_transform(chunk_texts)
            Q = vec.transform(query_texts)
            enc_time = time.perf_counter() - t_enc_start

            t_srch_start = time.perf_counter()
            scores = (Q @ C.T).toarray()
            srch_time = time.perf_counter() - t_srch_start
        else:
            C = encoder_model.encode(chunk_texts, normalize_embeddings=True)
            Q = encoder_model.encode(query_texts, normalize_embeddings=True)
            enc_time = time.perf_counter() - t_enc_start

            t_srch_start = time.perf_counter()
            scores = Q @ C.T
            srch_time = time.perf_counter() - t_srch_start

        for split in ["dev", "test"]:
            split_indices = [i for i, q in enumerate(queries) if q["split"] == split]
            split_scores = scores[split_indices]
            split_items = [queries[i] for i in split_indices]
            hit, mrr = evaluate_metrics(split_scores, chunks, split_items)
            print(f"{strat:<10} | {method:<10} | {split:<6} | {hit:<7.3f} | {mrr:<7.3f} | {len(chunks):<6} | {enc_time:<8.4f} | {srch_time:.4f}")

        # Log all individual rankings for CSV export
        for q_idx, q in enumerate(queries):
            row = scores[q_idx]
            top_ranked = np.argsort(-row, kind="stable")[:3]
            for rank, c_idx in enumerate(top_ranked, 1):
                c = chunks[c_idx]
                logs.append({
                    "strategy": strat,
                    "method": method,
                    "split": q["split"],
                    "query": q["query"],
                    "rank": rank,
                    "score": round(float(row[c_idx]), 4),
                    "chunk_id": c["id"],
                    "doc": c["doc"],
                    "start": c["start"],
                    "end": c["end"],
                    "text": c["text"],
                    "is_answerable": q["doc"] is not None
                })

# Save log to retrieval_log.csv
with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(logs[0].keys()))
    writer.writeheader()
    writer.writerows(logs)

print(f"\n[Logging] Successfully saved {len(logs)} detailed search rankings to 'retrieval_log.csv'.")


# ==============================================================================
# SECTION 4: DEMONSTRATION & FAILURE CASE ANALYSIS (REQUIRED EVIDENCE)
# ==============================================================================
print("\n" + "=" * 75)
print("FAILURE ANALYSIS: INSPECTION OF OBSERVED EDGE CASES")
print("=" * 75)

# Example 1: TF-IDF Lexical Miss on Paraphrase
q_para = "How can I recover my login access?"
c_para = make_chunks(docs, "sentence")
t_vec = TfidfVectorizer().fit([c["text"] for c in c_para])
sc_tfidf = (t_vec.transform([q_para]) @ t_vec.transform([c["text"] for c in c_para]).T).toarray()[0]
print("Edge Case 1 — Lexical Paraphrase Miss:")
print(f"  Query    : '{q_para}'")
print(f"  TF-IDF Max Score: {np.max(sc_tfidf):.3f} (Fails to match because no identical words exist with password document)")

# Example 2: Cosine Similarity on Unanswerable Query
q_unans = "Who won the football match?"
emb_q = encoder_model.encode([q_unans], normalize_embeddings=True)
emb_c = encoder_model.encode([c["text"] for c in c_para], normalize_embeddings=True)
sc_emb = (emb_q @ emb_c.T)[0]
top_idx = int(np.argmax(sc_emb))
print("\nEdge Case 2 — Unanswerable Query Spurious Retrieval:")
print(f"  Query           : '{q_unans}'")
print(f"  Top Candidate   : '{c_para[top_idx]['id']}' with Cosine Score: {sc_emb[top_idx]:.3f}")
print(f"  Evidence Retrieved: '{c_para[top_idx]['text'][:60]}...'")
print("  Explanation     : Dense embeddings compute relative angles in vector space; every query yields a top score even when semantically irrelevant.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.
[Setup] Loading SentenceTransformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Setup] Model loaded successfully.

TASK 1: TOKENIZATION COMPARISON (WHITESPACE vs. REGEX vs. SUBWORD)
Input Text    : 'AI, search!'
  Whitespace  ( 2): ['AI,', 'search!']
  Word-like   ( 2): ['AI', 'search']
  Subword     ( 4): ['ai', ',', 'search', '!']
---------------------------------------------------------------------------
Input Text    : "I can't log in."
  Whitespace  ( 4): ['I', "can't", 'log', 'in.']
  Word-like   ( 5): ['I', 'can', 't', 'log', 'in']
  Subword     ( 7): ['i', 'can', "'", 't', 'log', 'in', '.']
---------------------------------------------------------------------------
Input Text    : 'help@example.com'
  Whitespace  ( 1): ['help@example.com']
  Word-like   ( 3): ['help', 'example', 'com']
  Subword     ( 5): ['help', '@', 'example', '.', 'com']
---------------------------------------------------------------------------
Input Text    : 'CS4944 version 2.1'
  Whitespace  ( 3): ['CS4944', 'version', '2.1']
  Word-like   ( 4): ['CS4944', 'version', '2', '1']
  S